# Supplementary Figure S8: Multi-depth Receptive Fields vs. Single-depth Receptive Fields

This notebook generates the publication-ready supplementary figure comparing receptive fields measured under multi-depth stimulus conditions with standard single-depth receptive fields and retinotopy in V1:
- **A, B, C**: Example neurons representing Near, Mid, and Far depth tuning, showing their 1D depth tuning curves and side-by-side 2D spatial receptive fields (Single depth vs Multi depth) across 8 virtual depths.
- **D**: Population distribution of receptive field correlations between single-depth and multi-depth conditions (Contralateral vs. Ipsilateral control).
- **E**: Population yield: Proportion of V1 neurons with significant receptive fields across recording sessions.
- **F, G, H**: Spatial retinotopy and functional organization across an example V1 FOV: Azimuth retinotopy map (F), Elevation retinotopy map (G), and Preferred virtual depth map with anatomical two-photon mean image inset (H).

In [ ]:
%reload_ext autoreload
%autoreload 2

import pickle
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns
from scipy import stats

import flexiznam as flz
from cottage_analysis.analysis.spheres import rf_analysis, spheres
from cottage_analysis.plotting import depth_selectivity_plots, rf_plots, rsof_plots, plotting_utils, style
from cottage_analysis.pipelines import pipeline_utils
from v1_depth_map.revisions.revision_sessions import sessions
from v1_depth_map.paths import get_figures_roots

In [ ]:
# Register the manuscript font faces (Arial regular + bold + italic, Arial Narrow) and
# apply the publication rcParams: vector fonttypes, font sizes, tick/label padding.
# `style.savefig` then expands the SVG `font:` shorthand so Illustrator reads the
# family, size and weight correctly - see cottage_analysis.plotting.style for both.
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT

style.setup_figure_fonts()

In [ ]:
import os

# RF fits used by every RF panel: "neurons_df_spks.pickle" (RFs refitted on deconvolved
# spikes, used in the manuscript; other columns stay dF/F) or "neurons_df.pickle" (dF/F).
# Outputs and caches get a suffix.
RF_FILE = "neurons_df_spks.pickle"
OUT_SFX = "" if RF_FILE == "neurons_df.pickle" else "_" + RF_FILE[len("neurons_df_"):-len(".pickle")]
print(f"RF fits from {RF_FILE}, output suffix {OUT_SFX!r}")

project = "colasa_3d-vision_revisions"
flexilims_session = flz.get_flexilims_session(project)
READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
(SAVE_ROOT / "fig_supp_multidepth_rf").mkdir(parents=True, exist_ok=True)

## 1. Load Population & Session Data

In [ ]:
# Load all multi-depth revision sessions
multi_depth_sessions = [k for k, v in sessions.items() if v == "multidepth"]
print(f"Loading data for {len(multi_depth_sessions)} multi-depth sessions: {multi_depth_sessions}")

all_sig_m, all_sig_ipsi_m, neurons_df = rf_analysis.load_sig_rf(
    flexilims_session=flexilims_session,
    session_list=multi_depth_sessions,
    n_std=6,
    filter_datasets={"annotated": True},
    use_multidepth=True,
    use_cols=None,
    sphere_presentation_mask=None,
    rf_file=RF_FILE,
)

ndepths = 8
# Compute RF spatial centers and peak depth for single-depth protocol
coef_single = np.stack(neurons_df["rf_coef_closedloop"].values)
azi_s, ele_s, idepth_s, _ = rf_analysis.find_rf_centers(
    neurons_df, ndepths=ndepths, frame_shape=(16, 24), is_closed_loop=1, resolution=5, coef=coef_single
)
neurons_df["rf_azi_single"] = azi_s
neurons_df["rf_ele_single"] = ele_s
neurons_df["rf_idepth_single"] = idepth_s

# Compute RF spatial centers and peak depth for multi-depth protocol
coef_multi = np.stack(neurons_df["rf_coef_closedloop_multidepth"].values)
azi_m, ele_m, idepth_m, _ = rf_analysis.find_rf_centers(
    neurons_df, ndepths=ndepths, frame_shape=(16, 24), is_closed_loop=1, resolution=5, coef=coef_multi
)
neurons_df["rf_azi_multi"] = azi_m
neurons_df["rf_ele_multi"] = ele_m
neurons_df["rf_idepth_multi"] = idepth_m

# Calculate pixel-by-pixel full-volume correlation between single-depth and multi-depth RFs
def get_rf_corr(row):
    try:
        c_single = row["rf_coef_closedloop"][:, :-1].flatten()
        c_multi = row["rf_coef_closedloop_multidepth"][:, :-1].flatten()
        valid = ~np.isnan(c_single) & ~np.isnan(c_multi)
        if valid.sum() > 10:
            return np.corrcoef(c_single[valid], c_multi[valid])[0, 1]
        return np.nan
    except Exception:
        return np.nan

def get_rf_ipsi_corr(row):
    try:
        c_single = row["rf_coef_ipsi_closedloop"][:, :-1].flatten()
        c_multi = row["rf_coef_ipsi_closedloop_multidepth"][:, :-1].flatten()
        valid = ~np.isnan(c_single) & ~np.isnan(c_multi)
        if valid.sum() > 10:
            return np.corrcoef(c_single[valid], c_multi[valid])[0, 1]
        return np.nan
    except Exception:
        return np.nan

neurons_df["rf_corr"] = neurons_df.apply(get_rf_corr, axis=1)
neurons_df["rf_ipsi_corr"] = neurons_df.apply(get_rf_ipsi_corr, axis=1)

# Identify depth-tuned neurons
is_depth_tuned = (neurons_df["iscell"] == 1) & (
    depth_selectivity_plots.common_utils.one_sided_pval_from_spearman(
        neurons_df["depth_tuning_test_spearmanr_rval_closedloop"],
        neurons_df["depth_tuning_test_spearmanr_pval_closedloop"],
    )
    < 0.05
)
neurons_df["is_depth_tuned"] = is_depth_tuned
sig_depth_df = neurons_df[(neurons_df["rf_sig"] == 1) & (neurons_df["is_depth_tuned"])].copy()
print(f"Loaded {len(neurons_df)} total neurons, {len(sig_depth_df)} depth-tuned with significant RF.")

In [ ]:
# Load example session data for tuning curves and RF maps
example_session = "PZAH17.1e_S20250318"
neurons_df_example = pipeline_utils.load_neurons_df(
    pipeline_utils.create_neurons_ds(
        session_name=example_session,
        flexilims_session=flexilims_session,
        project=None,
        conflicts="skip",
    ),
    rf_file=RF_FILE,
)
_, trials_df_example = spheres.sync_all_recordings(
    session_name=example_session,
    flexilims_session=flexilims_session,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base="SpheresPermTubeReward",
    photodiode_protocol=5,
    return_volumes=True,
)
depths = np.sort(trials_df_example.depth.unique())
print(f"Example session {example_session} loaded with {len(neurons_df_example)} neurons.")

In [ ]:
# Load population RF significance across all recording sessions (Panel E)
# written by figsupp9_v1_depth_map (with the same RF_FILE)
with open(READ_ROOT / f"rf_all_sig{OUT_SFX}.pkl", "rb") as f:
    all_sig = pickle.load(f)
with open(READ_ROOT / f"rf_all_sig_ipsi{OUT_SFX}.pkl", "rb") as f:
    all_sig_ipsi = pickle.load(f)
print(f"Loaded RF summary for {len(all_sig)} sessions. Median sig RF proportion: {np.median(all_sig):.2%}")
# Neuron table all_sig was computed from (same cache, same RF_FILE), for the Panel E
# neuron and mouse counts
neurons_df_single = pd.read_pickle(READ_ROOT / f"rf_neurons_df_slim{OUT_SFX}.pkl")

# Load example FOV session and suite2p ROI metadata (Panels F–H)
session_name_fov = "PZAH8.2f_S20230206"
project_rf = "hey2_3d-vision_foodres_20220101"
flexilims_session_rf = flz.get_flexilims_session(project_rf)
neurons_ds_fov = pipeline_utils.create_neurons_ds(
    session_name=session_name_fov,
    flexilims_session=flexilims_session_rf,
    project=project_rf,
    conflicts="skip",
)
neurons_df_fov = pipeline_utils.load_neurons_df(neurons_ds_fov, rf_file=RF_FILE)
suite2p_ds_fov = flz.get_datasets(
    flexilims_session=flexilims_session_rf,
    origin_name=session_name_fov,
    dataset_type="suite2p_rois",
    filter_datasets={"anatomical_only": 3},
    allow_multiple=False,
    return_dataseries=False,
)
neurons_df_fov["iscell"] = np.load(suite2p_ds_fov.path_full / "plane0" / "iscell.npy", allow_pickle=True)[:, 0]
stat_fov = np.load(suite2p_ds_fov.path_full / "plane0" / "stat.npy", allow_pickle=True)
ops_fov = np.load(suite2p_ds_fov.path_full / "plane0" / "ops.npy", allow_pickle=True).item()
print(f"Example FOV session {session_name_fov} loaded with {len(neurons_df_fov)} neurons.")

## 2. Layout Configuration & Helper Functions

In [ ]:
FIG_W = 19.0   # cm
FIG_H = 18.8   # cm (increased to integrate bottom FOV row)
Y_SHIFT = 7.1  # cm upward shift for top rows A–E

# Centimetre-based layout helpers, shared so panel letters stay at
# FONTSIZE_DICT["panel"] across every figure.
from cottage_analysis.plotting.style import rect_cm, panel_letter

## 3. Panels A–C: Example Neurons (Single-Depth vs. Multi-Depth)

In [ ]:
# Example neurons representing Near, Mid, and Far depth tuning (distinct from main fig ROIs 53, 118, 623)
EXAMPLE_ROIS = [210, 186, 497]

def plot_example_panels(fig, neurons_df, trials_df, rois=EXAMPLE_ROIS, y_shift=Y_SHIFT):
    col_w = 3.4
    col_gap = 0.5
    rf_h = 7.7  # height of RF stack
    plot_prop = 0.90
    h_slice = rf_h / ndepths * plot_prop
    rf_w = 1.5 * h_slice
    rf_gap = 0.10
    tune_w = 2 * rf_w + rf_gap  # Matched to 2-column matrix width
    
    for i_ex, roi in enumerate(rois):
        x_base = 0.6 + i_ex * (col_w + col_gap)
        
        # Depth tuning curve (black trace & points, large marker size)
        ax_tune = fig.add_axes(rect_cm(fig, x_base + 0.1, 9.7 + y_shift, tune_w, 1.3))
        depth_selectivity_plots.plot_depth_tuning_curve(
            neurons_df=neurons_df,
            trials_df=trials_df,
            roi=roi,
            linecolor="k",
            fit_linecolor="k",
            markeredgecolor="k",
            rs_thr=None,
            plot_fit=True,
            plot_smooth=False,
            linewidth=1.2,
            closed_loop=1,
            fontsize_dict=FONTSIZE_DICT,
            markersize=6.5,
            ylim_precision_base=5,
            ylim_precision=2,
        )
        ax_tune.set_ylabel(r"$\Delta F/F_0$" if i_ex == 0 else "", fontsize=FONTSIZE_DICT["label"])
        ax_tune.set_xlabel("Virtual depth (cm)", fontsize=FONTSIZE_DICT["label"], labelpad=1)
        xticks = ax_tune.get_xticks()
        xlabels = [t.get_text() for t in ax_tune.get_xticklabels()]
        ax_tune.set_xticks(xticks[::2], xlabels[::2], rotation=45, fontsize=FONTSIZE_DICT["tick"])
        ax_tune.tick_params(length=1.5, pad=1)
        
        # Compute shared clim for consistent scaling across single and multi depth
        c_s = neurons_df.loc[roi, "rf_coef_closedloop"][:, :-1]
        c_m = neurons_df.loc[roi, "rf_coef_closedloop_multidepth"][:, :-1]
        c_s_mean = np.nanmean(c_s.reshape(c_s.shape[0], ndepths, 16, 24), axis=0)
        c_m_mean = np.nanmean(c_m.reshape(c_m.shape[0], ndepths, 16, 24), axis=0)
        clim = max(np.nanmax(np.abs(c_s_mean)), np.nanmax(np.abs(c_m_mean)), 0.1)
        # Round up to 0.1: plot_rf ticks the colorbar at +/- round(clim, 1), so the
        # end ticks only sit at the ends of the bar if clim is already rounded
        clim = np.ceil(clim * 10) / 10
        
        # Extended Single depth RF stack (8 depths) - no colorbar, title on top axis
        axes_s = rf_plots.plot_rf(
            neurons_df=neurons_df,
            roi=roi,
            ndepths=ndepths,
            frame_shape=(16, 24),
            position=rect_cm(fig, x_base + 0.1, 7.6 + y_shift, rf_w, rf_h),
            plot_prop=plot_prop,
            xlabel="",
            ylabel="Elevation (°)" if i_ex == 0 else "",
            fontsize_dict=FONTSIZE_DICT,
            use_multidepth=False,
            clim=clim,
            plot_yticklabels=(i_ex == 0),
            colorbar=False,
        )
        axes_s[0].set_title("Single depth", fontsize=FONTSIZE_DICT["label"], pad=2)
        
        # Extended Multi depth RF stack (8 depths) - tight next to single depth, keep shared colorbar
        axes_m = rf_plots.plot_rf(
            neurons_df=neurons_df,
            roi=roi,
            ndepths=ndepths,
            frame_shape=(16, 24),
            position=rect_cm(fig, x_base + 0.1 + rf_w + rf_gap, 7.6 + y_shift, rf_w, rf_h),
            plot_prop=plot_prop,
            xlabel="",
            ylabel="",
            fontsize_dict=FONTSIZE_DICT,
            use_multidepth=True,
            clim=clim,
            plot_yticklabels=False,
            colorbar=True,
        )
        axes_m[0].set_title("Multi depth", fontsize=FONTSIZE_DICT["label"], pad=2)
        
        # Single centered Azimuth label per panel (A, B, C)
        x_center_rf = x_base + 0.1 + rf_w + rf_gap / 2
        fig.text(
            x_center_rf / FIG_W,
            (0.28 + y_shift) / FIG_H,
            "Azimuth (°)",
            ha="center",
            va="bottom",
            fontsize=FONTSIZE_DICT["label"],
        )

## 4. Panels D–E: Population Similarity & RF Yield Metrics

In [ ]:
def plot_population_panels(fig, df, all_sig, all_sig_ipsi, all_sig_multi, y_shift=Y_SHIFT):
    # Only include neurons with significant RF
    df_rf = df[df["rf_sig"] == 1].copy() if "rf_sig" in df.columns else df.copy()

    # Panel D: Top of right third (RF Correlation Histogram vs Ipsi Control), for the
    # depth-tuned neurons with a significant RF whose contra and ipsi correlations are
    # both defined
    ax_d = fig.add_axes(rect_cm(fig, 13.2, 8.4 + y_shift, 4.2, 2.6))
    corr_df = df_rf[df_rf["is_depth_tuned"]].dropna(subset=["rf_corr", "rf_ipsi_corr"])
    corr_contra = corr_df["rf_corr"]
    corr_ipsi = corr_df["rf_ipsi_corr"]
    
    bins = np.linspace(-0.4, 1.0, 25)
    ax_d.hist(corr_contra, bins=bins, density=True, alpha=0.6, color="royalblue", label="Contralateral", edgecolor="none")
    ax_d.hist(corr_ipsi, bins=bins, density=True, alpha=0.5, color="gray", label="Ipsilateral", edgecolor="none")
    ax_d.axvline(corr_contra.median(), color="blue", linestyle="--", linewidth=1.2)
    ax_d.axvline(corr_ipsi.median(), color="dimgray", linestyle="--", linewidth=1.2)
    ax_d.set_xlabel("RF correlation (Pearson r)", fontsize=FONTSIZE_DICT["label"])
    ax_d.set_ylabel("Density", fontsize=FONTSIZE_DICT["label"])
    ax_d.legend(
        fontsize=FONTSIZE_DICT["legend"],
        frameon=False,
        loc="lower left",
        bbox_to_anchor=(0.0, 1.02),
        ncol=2,
        handletextpad=0.4,
        columnspacing=0.8,
    )
    ax_d.set_xlim(-0.4, 1.0)
    ax_d.set_xticks([-0.4, 0.0, 0.5, 1.0])
    ax_d.set_yticks([0, 1, 2, 3, 4, 5])
    ax_d.tick_params(labelsize=FONTSIZE_DICT["tick"], length=1.5)
    sns.despine(ax=ax_d, trim=True)
    
    # Panel E: Bottom of right third (Proportion of significant RFs histogram across sessions)
    ax_e = fig.add_axes(rect_cm(fig, 13.2, 5.2 + y_shift, 4.2, 1.9))
    rf_plots.plot_sig_rf_perc(
        all_sig=all_sig,
        all_sig_ipsi=all_sig_ipsi,
        plot_type="hist",
        hist_color="cornflowerblue",
        hist_edgecolor="royalblue",
        bins=np.arange(0, 1, 0.1),
        fontsize_dict=FONTSIZE_DICT,
    )
    # Multi-depth sessions stacked on top of the single-depth ones, same bins
    multi_color = "darkorchid"
    bins_e = np.arange(0, 1, 0.1)
    n_single, _ = np.histogram(all_sig, bins=bins_e)
    n_multi, _ = np.histogram(all_sig_multi, bins=bins_e)
    ax_e.bar(bins_e[:-1], n_multi, width=np.diff(bins_e), bottom=n_single, align="edge", color=multi_color, alpha=0.6, edgecolor=multi_color, linewidth=1)
    # Black outline of the total (single + multi-depth) histogram
    n_total = n_single + n_multi
    ax_e.stairs(n_total, bins_e, color="k", linewidth=1)
    # Rescale y to the stacked counts. Drop the single-depth median triangle
    # (last line drawn by plot_sig_rf_perc): only the median of all sessions is shown
    # headroom so the median triangle (at 95% of ymax) clears the tallest bar
    ymax = np.ceil(np.max(n_total) / 0.85 / 5) * 5
    ax_e.lines[-1].remove()
    ax_e.set_ylim(0, ymax)
    all_sessions = np.concatenate([all_sig, all_sig_multi])
    ax_e.plot(np.median(all_sessions), ymax * 0.95, marker="v", markersize=7, color="k", markeredgecolor="k", markeredgewidth=1)
    ax_e.legend(
        handles=[
            plt.Rectangle((0, 0), 1, 1, facecolor="cornflowerblue", edgecolor="royalblue"),
            plt.Rectangle((0, 0), 1, 1, facecolor=multi_color, alpha=0.6, edgecolor=multi_color),
        ],
        labels=[f"Single depth (n = {len(all_sig)})", f"Multi-depth (n = {len(all_sig_multi)})"],
        fontsize=FONTSIZE_DICT["legend"],
        frameon=False,
        loc="upper right",
        handlelength=1.0,
        handletextpad=0.4,
    )
    ax_e.tick_params(labelsize=FONTSIZE_DICT["tick"], length=1.5)

    # Panel F: single- vs multi-depth RF peak depth, below E
    # Returns the neurons of panels D and F, for the panel statistics
    return corr_df, plot_peak_depth_scatter(fig, df_rf, y_shift=y_shift)

def plot_peak_depth_scatter(fig, df_rf, y_shift=Y_SHIFT):
    """Single-depth vs multi-depth RF peak depth (cm) of depth-tuned neurons.

    Returns the plotted neurons' peak depths (m), for the panel statistics.
    """
    ax_scatter = fig.add_axes(rect_cm(fig, 13.7, 0.45 + y_shift, 3.45, 3.45))
    df_depth = df_rf[df_rf["is_depth_tuned"]] if "is_depth_tuned" in df_rf.columns else df_rf
    valid_pref_d = df_depth[["rf_preferred_depth_closedloop", "rf_preferred_depth_closedloop_multidepth"]].dropna()
    
    x_cm = valid_pref_d["rf_preferred_depth_closedloop"] * 100
    y_cm = valid_pref_d["rf_preferred_depth_closedloop_multidepth"] * 100
    
    xlim = [1.5, 2500]
    ylim = [1.5, 2500]  # RF peak depths span 2-2000 cm on both axes
    ax_scatter.scatter(x_cm, y_cm, s=8, alpha=0.35, color="darkorchid", edgecolors="none", clip_on=False)
    ax_scatter.set_xscale("log")
    ax_scatter.set_yscale("log")
    ax_scatter.plot([ylim[0], xlim[1]], [ylim[0], xlim[1]], "k--", linewidth=0.8, alpha=0.7)
    ax_scatter.set_xlim(xlim)
    ax_scatter.set_ylim(ylim)
    ax_scatter.set_xticks([10, 100, 1000])
    ax_scatter.set_xticklabels(["10", "100", "1000"])
    ax_scatter.set_yticks([10, 100, 1000])
    ax_scatter.set_yticklabels(["10", "100", "1000"])
    ax_scatter.set_aspect("equal")
    ax_scatter.tick_params(which="major", length=3.0, labelsize=FONTSIZE_DICT["tick"])
    ax_scatter.tick_params(which="minor", length=1.5)
    ax_scatter.set_xlabel("Single-depth RF peak depth (cm)", fontsize=FONTSIZE_DICT["label"])
    ax_scatter.set_ylabel("Multi-depth RF peak depth (cm)", fontsize=FONTSIZE_DICT["label"])
    sns.despine(ax=ax_scatter)
    ax_scatter.spines["bottom"].set_bounds(xlim[0], xlim[1])
    ax_scatter.spines["left"].set_bounds(ylim[0], ylim[1])
    return valid_pref_d

## 5. Panels F–H: Spatial Retinotopy in Example V1 FOV

In [ ]:
def plot_fov_panels(fig, neurons_df_fov, ops_fov, stat_fov):
    fov_w = 5.92
    fov_h = 5.92
    fov_y = 0.45
    
    # Panel F: Azimuth map across example FOV
    fig.add_axes(rect_cm(fig, 0.25, fov_y, fov_w, fov_h))
    depth_selectivity_plots.plot_example_fov(
        neurons_df=neurons_df_fov,
        ops=ops_fov,
        stat=stat_fov,
        ndepths=ndepths,
        col="rf_azi",
        cmap=cm.YlOrRd.reversed(),
        background_color=np.array([0, 0, 0]),
        fontsize_dict=FONTSIZE_DICT,
        fov_width=661,
    )
    
    # Panel G: Elevation map across example FOV
    fig.add_axes(rect_cm(fig, 6.45, fov_y, fov_w, fov_h))
    depth_selectivity_plots.plot_example_fov(
        neurons_df=neurons_df_fov,
        ops=ops_fov,
        stat=stat_fov,
        ndepths=ndepths,
        col="rf_ele",
        cmap=cm.YlOrRd.reversed(),
        background_color=np.array([0, 0, 0]),
        fontsize_dict=FONTSIZE_DICT,
        fov_width=661,
    )
    
    # Panel H: Preferred Depth map across example FOV + anatomical meanImg inset
    fig.add_axes(rect_cm(fig, 12.65, fov_y, fov_w, fov_h))
    depth_selectivity_plots.plot_example_fov(
        neurons_df=neurons_df_fov,
        ops=ops_fov,
        stat=stat_fov,
        ndepths=ndepths,
        col="preferred_depth_closedloop",
        cmap=cm.cool.reversed(),
        background_color=np.array([0, 0, 0]),
        fontsize_dict=FONTSIZE_DICT,
        fov_width=661,
    )
    
    # Inset for FOV mean image at top-right of panel H
    ins_w = 1.85
    ins_h = 1.85
    x_ins = 12.65 + fov_w - ins_w - 0.05
    y_ins = fov_y + fov_h - ins_h - 0.05
    
    # White background patch
    ax_bg = fig.add_axes(rect_cm(fig, x_ins - 0.06, y_ins - 0.06, ins_w + 0.12, ins_h + 0.12))
    ax_bg.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor="white", edgecolor="none"))
    ax_bg.set_axis_off()
    
    # Mean image inset
    fig.add_axes(rect_cm(fig, x_ins, y_ins, ins_w, ins_h))
    depth_selectivity_plots.plot_fov_mean_img(ops_fov["meanImg"], fov_width=661)

## 6. Assemble and Save Unified Publication Figure

In [ ]:
# Assemble unified publication canvas
fig = plt.figure(figsize=(FIG_W / 2.54, FIG_H / 2.54), dpi=300)
bg_ax = fig.add_axes([0, 0, 1, 1])
bg_ax.set_xticks([])
bg_ax.set_yticks([])
fig.patch.set_facecolor("white")

# Panel letters
panel_letter(fig, "A", 0.1, 11.2 + Y_SHIFT)
panel_letter(fig, "B", 3.6, 11.2 + Y_SHIFT)
panel_letter(fig, "C", 7.5, 11.2 + Y_SHIFT)
panel_letter(fig, "D", 12.3, 11.2 + Y_SHIFT)
panel_letter(fig, "E", 12.3, 7.5 + Y_SHIFT)
panel_letter(fig, "F", 12.3, 4.2 + Y_SHIFT)

panel_letter(fig, "G", 0.1, 6.6)
panel_letter(fig, "H", 6.2, 6.6)
panel_letter(fig, "I", 12.4, 6.6)

# Draw Panels
plot_example_panels(fig, neurons_df_example, trials_df_example)
corr_df, peak_depth_df = plot_population_panels(fig, neurons_df, all_sig, all_sig_ipsi, all_sig_m)
plot_fov_panels(fig, neurons_df_fov, ops_fov, stat_fov)

# Export publication-ready vector and raster formats
save_path = SAVE_ROOT / "fig_supp_multidepth_rf" / f"figsupp_multidepth_receptive_fields{OUT_SFX}"
for ext in ["pdf", "svg", "png"]:
    style.savefig(f"{save_path}.{ext}", fig=fig, dpi=300, bbox_inches="tight")
print(f"Supplementary Figure successfully generated and saved to:\n  {save_path}.png/.pdf/.svg")
plt.show()

In [ ]:
# Export statistics for manuscript tracking (panels D, E and F).
# Proportion of depth-tuned neurons with a significant RF per session, as plotted in E:
# single-depth sessions (all_sig, from figsupp9_v1_depth_map), multi-depth sessions
# (all_sig_m, multi-depth RFs) and both together (the black triangle).
from v1_depth_map.stats import export_figure_stats

stats_supp8 = {}

# Panel D: correlation of the single- and multi-depth RF volumes, contra vs ipsi
median_contra = float(corr_df["rf_corr"].median())
median_ipsi = float(corr_df["rf_ipsi_corr"].median())
_, p_corr = stats.mannwhitneyu(corr_df["rf_corr"], corr_df["rf_ipsi_corr"])
mantissa, exponent = f"{p_corr:.2e}".split("e")
n_corr_sessions = corr_df["session"].nunique()
n_corr_mice = corr_df["session"].str.split("_").str[0].nunique()
stats_supp8.update(
    {
        "rf_corr_n_neurons": {
            "raw": len(corr_df),
            "formatted": f"{len(corr_df):,}",
            "latex_macro": "statSuppEightRFCorrNeurons",
            "description": "Depth-tuned neurons with significant multi-depth RFs in the RF correlation comparison (Fig S8D)",
        },
        "rf_corr_n_sessions": {
            "raw": int(n_corr_sessions),
            "formatted": f"{n_corr_sessions}",
            "latex_macro": "statSuppEightRFCorrSessions",
            "description": "Number of multi-depth sessions in the RF correlation comparison (Fig S8D)",
        },
        "rf_corr_n_mice": {
            "raw": int(n_corr_mice),
            "formatted": f"{n_corr_mice}",
            "latex_macro": "statSuppEightRFCorrMice",
            "description": "Number of mice in the RF correlation comparison (Fig S8D)",
        },
        "rf_corr_median_contra": {
            "raw": median_contra,
            "formatted": f"$r = {median_contra:.3f}$",
            "latex_macro": "statSuppEightRFCorrMedianContra",
            "description": "Median Pearson r between single- and multi-depth contralateral RF volumes (Fig S8D)",
        },
        "rf_corr_median_ipsi": {
            "raw": median_ipsi,
            "formatted": f"$r = {median_ipsi:.3f}$",
            "latex_macro": "statSuppEightRFCorrMedianIpsi",
            "description": "Median Pearson r between single- and multi-depth ipsilateral RF volumes (Fig S8D)",
        },
        "rf_corr_mannwhitney_p": {
            "raw": float(p_corr),
            "formatted": f"$p = {mantissa} \\times 10^{{{int(exponent)}}}$",
            "latex_macro": "statSuppEightRFCorrPval",
            "description": "Mann-Whitney U test p-value, contralateral vs ipsilateral RF correlations (Fig S8D)",
        },
    }
)

for key, label, macro, values in [
    ("single", "single-depth", "Single", all_sig),
    ("multi", "multi-depth", "Multi", all_sig_m),
    ("total", "single- and multi-depth", "Total", np.concatenate([all_sig, all_sig_m])),
]:
    n, median = len(values), float(np.median(values))
    stats_supp8[f"sig_rf_n_sessions_{key}"] = {
        "raw": n,
        "formatted": f"{n}",
        "latex_macro": f"statSuppEightSigRFSessions{macro}",
        "description": f"Number of {label} sessions (Fig S8E)",
    }
    stats_supp8[f"sig_rf_median_{key}"] = {
        "raw": median,
        "formatted": f"{median * 100:.1f}\\%",
        "latex_macro": f"statSuppEightSigRFMedian{macro}",
        "description": f"Median proportion of depth-tuned neurons with significant RFs across {label} sessions (Fig S8E)",
    }
# Neurons behind the proportions: depth-tuned cells of each session, as in
# rf_analysis.load_sig_rf. They must give back the proportions plotted in E.
from cottage_analysis.analysis import common_utils

single_cells = neurons_df_single[
    (neurons_df_single["iscell"] == 1)
    & (
        common_utils.one_sided_pval_from_spearman(
            neurons_df_single["depth_tuning_test_spearmanr_rval_closedloop"],
            neurons_df_single["depth_tuning_test_spearmanr_pval_closedloop"],
        )
        < 0.05
    )
]
multi_cells = neurons_df[neurons_df["is_depth_tuned"]]
for cells, values in [(single_cells, all_sig), (multi_cells, all_sig_m)]:
    props = cells.groupby("session", sort=False)["rf_sig"].mean().to_numpy()
    assert np.allclose(props, values), "neuron table does not match the panel E proportions"
total_cells = pd.concat([single_cells[["session", "rf_sig"]], multi_cells[["session", "rf_sig"]]])
n_mice_total = total_cells["session"].str.split("_").str[0].nunique()
n_neurons_total = len(total_cells)
n_sig_total = int((total_cells["rf_sig"] == 1).sum())
stats_supp8["sig_rf_n_mice_total"] = {
    "raw": int(n_mice_total),
    "formatted": f"{n_mice_total}",
    "latex_macro": "statSuppEightSigRFMiceTotal",
    "description": "Number of mice, single- and multi-depth sessions (Fig S8E)",
}
stats_supp8["sig_rf_n_neurons_total"] = {
    "raw": n_neurons_total,
    "formatted": f"{n_neurons_total:,}",
    "latex_macro": "statSuppEightSigRFNeuronsTotal",
    "description": "Depth-tuned neurons, single- and multi-depth sessions (Fig S8E)",
}
stats_supp8["sig_rf_n_sig_neurons_total"] = {
    "raw": n_sig_total,
    "formatted": f"{n_sig_total:,}",
    "latex_macro": "statSuppEightSigRFSigNeuronsTotal",
    "description": "Depth-tuned neurons with significant RFs, single- and multi-depth sessions (Fig S8E)",
}

# Panel F: multi- vs single-depth RF peak depth of the plotted neurons
r_peak, p_peak = stats.spearmanr(
    peak_depth_df["rf_preferred_depth_closedloop"],
    peak_depth_df["rf_preferred_depth_closedloop_multidepth"],
)
ratio = (
    peak_depth_df["rf_preferred_depth_closedloop_multidepth"]
    / peak_depth_df["rf_preferred_depth_closedloop"]
)
q1, median_ratio, q3 = np.percentile(ratio, [25, 50, 75])
mantissa, exponent = f"{p_peak:.2e}".split("e")
stats_supp8.update(
    {
        "peak_depth_n_neurons": {
            "raw": len(peak_depth_df),
            "formatted": f"{len(peak_depth_df):,}",
            "latex_macro": "statSuppEightPeakDepthNeurons",
            "description": "Depth-tuned neurons with significant multi-depth RFs in the peak depth comparison (Fig S8F)",
        },
        "peak_depth_spearman_r": {
            "raw": float(r_peak),
            "formatted": f"$r = {r_peak:.3f}$",
            "latex_macro": "statSuppEightPeakDepthR",
            "description": "Spearman correlation of multi- vs single-depth RF peak depth (Fig S8F)",
        },
        "peak_depth_spearman_p": {
            "raw": float(p_peak),
            "formatted": f"$p = {mantissa} \\times 10^{{{int(exponent)}}}$",
            "latex_macro": "statSuppEightPeakDepthPval",
            "description": "p-value of the Spearman correlation of multi- vs single-depth RF peak depth (Fig S8F)",
        },
        "peak_depth_median_ratio": {
            "raw": float(median_ratio),
            "formatted": f"{median_ratio:.3f}",
            "latex_macro": "statSuppEightPeakDepthMedianRatio",
            "description": "Median ratio of multi- to single-depth RF peak depth (Fig S8F)",
        },
        "peak_depth_ratio_iqr": {
            "raw": [float(q1), float(q3)],
            "formatted": f"{q1:.3f}--{q3:.3f}",
            "latex_macro": "statSuppEightPeakDepthRatioIQR",
            "description": "Interquartile range of the multi- to single-depth RF peak depth ratio (Fig S8F)",
        },
    }
)

if OUT_SFX == "_spks":  # the manuscript uses the RFs fitted on deconvolved spikes
    export_figure_stats("figsupp8_multidepth_receptive_fields", stats_supp8, figure="supp")
else:
    print("RF variant: manuscript stats not exported")

for k, v in stats_supp8.items():
    print(f"{k}: {v['raw']} ({v['formatted']}) - {v['description']}")


## Alternative Example FOV Sessions: PZAH8.2h_S20230302 & PZAH8.2f_S20230117

Plots the 3-panel FOV retinotopic organization (Azimuth, Elevation, and Preferred Depth with anatomical mean image inset) for the other two candidate single-depth sessions:
- **PZAH8.2h_S20230302**: 54.1% significant RF yield among depth-tuned neurons (168 significant RF cells out of 556 ROIs, from mouse PZAH8.2h).
- **PZAH8.2f_S20230117**: 45.5% significant RF yield among depth-tuned neurons (128 significant RF cells out of 478 ROIs, from mouse PZAH8.2f).

In [ ]:
# Plot FOV panels for the other two suggested candidate sessions
alt_sessions = [
    ("PZAH8.2h_S20230302", "54.1% sig RF yield"),
    ("PZAH8.2f_S20230117", "45.5% sig RF yield"),
]
save_dir = SAVE_ROOT / "fig_supp_multidepth_rf"
save_dir.mkdir(parents=True, exist_ok=True)

for alt_sess, yield_str in alt_sessions:
    print(f"Plotting FOV panels for alternative session: {alt_sess} ({yield_str})...")
    neurons_ds_alt = pipeline_utils.create_neurons_ds(
        session_name=alt_sess,
        flexilims_session=flexilims_session_rf,
        project=project_rf,
        conflicts="skip",
    )
    neurons_df_alt = pipeline_utils.load_neurons_df(neurons_ds_alt, rf_file=RF_FILE)
    suite2p_ds_alt = flz.get_datasets(
        flexilims_session=flexilims_session_rf,
        origin_name=alt_sess,
        dataset_type="suite2p_rois",
        filter_datasets={"anatomical_only": 3},
        allow_multiple=False,
        return_dataseries=False,
    )
    neurons_df_alt["iscell"] = np.load(
        suite2p_ds_alt.path_full / "plane0" / "iscell.npy", allow_pickle=True
    )[:, 0]
    stat_alt = np.load(
        suite2p_ds_alt.path_full / "plane0" / "stat.npy", allow_pickle=True
    )
    ops_alt = np.load(
        suite2p_ds_alt.path_full / "plane0" / "ops.npy", allow_pickle=True
    ).item()

    fig_alt = plt.figure(figsize=(FIG_W / 2.54, 7.0 / 2.54), dpi=300)
    bg_ax = fig_alt.add_axes([0, 0, 1, 1])
    bg_ax.set_xticks([])
    bg_ax.set_yticks([])
    fig_alt.patch.set_facecolor("white")

    fig_alt.text(
        0.5 / FIG_W,
        6.6 / 7.0,
        f"Session: {alt_sess} ({yield_str})",
        fontsize=FONTSIZE_DICT["title"],
        fontweight="bold",
        ha="left",
        va="top",
    )

    plot_fov_panels(fig_alt, neurons_df_alt, ops_alt, stat_alt)

    style.savefig(
        save_dir / f"figsupp_example_fov_{alt_sess}{OUT_SFX}.png",
        fig=fig_alt,
        dpi=300,
        bbox_inches="tight",
    )
    style.savefig(
        save_dir / f"figsupp_example_fov_{alt_sess}{OUT_SFX}.pdf",
        fig=fig_alt,
        bbox_inches="tight",
    )
    style.savefig(
        save_dir / f"figsupp_example_fov_{alt_sess}{OUT_SFX}.svg",
        fig=fig_alt,
        bbox_inches="tight",
    )
    plt.show()

## Retinotopic and Depth Gradients across Single-Depth V1 FOVs

For every single-depth V1 session of `hey2_3d-vision_foodres_20220101`, fits RF azimuth, RF elevation and log2 preferred depth as a plane over the cortical position of the selected neurons (depth-tuned, iscell, significant RF, as in Fig. 4 I-K; sessions with fewer than 10 such neurons are skipped):
- **Top left**: distribution across sessions of the $R^2$ of the gradient fit (triangles: medians).
- **Top right**: proportion of sessions whose gradient beats a shuffle of cells across positions (p < 0.05, 10,000 shuffles; dashed line: chance). Neighbouring cells are spatially correlated, so these p-values are optimistic.
- **Bottom**: gradient of each session as an arrow from the origin, length in map units per 100 µm, coloured if significant and light grey if not. Oriented as on the FOV maps (x flipped, y down).

In [ ]:
from v1_depth_map.figure_utils import fov_gradients

recompute_fov_gradients = False  # True: refit every session and re-cache; False: load the cached fits

_, SAVE_ROOT_RF = get_figures_roots(flexilims_session_rf)
gradients_cache = SAVE_ROOT_RF / "rf_supp" / f"fov_gradients{OUT_SFX}.pkl"
gradients_df = fov_gradients.load_fov_gradients(
    flexilims_session_rf,
    gradients_cache,
    recompute=recompute_fov_gradients or not gradients_cache.exists(),
    rf_file=RF_FILE,
)
gradients_df["sig"] = gradients_df["pval_perm"] < 0.05
print(
    gradients_df.groupby("map", sort=False).agg(
        n_sessions=("session", "size"),
        median_neurons=("n", "median"),
        median_r2=("r2", "median"),
        n_sig=("sig", "sum"),
        prop_sig=("sig", "mean"),
        median_gradient_per_100um=("magnitude", "median"),
    )
)

fig = plt.figure(figsize=(12 / 2.54, 10 / 2.54), dpi=300)
fov_gradients.plot_fov_gradient_panels(
    fig,
    rect_cm(fig, 1.2, 6.0, 5.5, 3.6),
    rect_cm(fig, 9.0, 6.0, 2.4, 3.6),
    gradients_df,
    FONTSIZE_DICT,
)
fov_gradients.plot_fov_gradient_arrows(
    fig,
    [rect_cm(fig, 0.4 + i * 3.9, 0.8, 3.3, 3.3) for i in range(3)],
    gradients_df,
    FONTSIZE_DICT,
)
save_path = SAVE_ROOT / "fig_supp_multidepth_rf" / f"figsupp_fov_gradients{OUT_SFX}"
for ext in ["pdf", "svg", "png"]:
    style.savefig(f"{save_path}.{ext}", fig=fig, dpi=300, bbox_inches="tight")


### Gradients at each FOV's location in V1

Same per-session plane fits, refitted on the neuron positions aligned across mice for Supplementary Figure 9 (`overview_x_aligned`, `overview_y_aligned`, from the `figsupp9_v1_depth_map` neuron cache), with the same neuron selection. The alignment only shifts each mouse, so $R^2$ and p-values are those above, but the gradients are now in cortical orientation. Each session's arrow is centred on its FOV, in the frame of Supplementary Figure 9 B-D; arrows are coloured if significant and grey if not, with length proportional to the gradient magnitude (scale under each map).

In [ ]:
from cottage_analysis.analysis import common_utils

# Neuron table aligned across mice, written by figsupp9_v1_depth_map (same RF_FILE),
# and its selection: depth-tuned cells with a significant RF in V1 sessions
READ_ROOT_RF, _ = get_figures_roots(flexilims_session_rf)
neurons_df_v1 = pd.read_pickle(READ_ROOT_RF / f"rf_neurons_df_slim{OUT_SFX}.pkl")
common_utils.add_one_sided_spearman_significance(
    neurons_df_v1,
    rval_col="depth_tuning_test_spearmanr_rval_closedloop",
    pval_col="depth_tuning_test_spearmanr_pval_closedloop",
    out_col="is_depth_neuron",
)
neurons_df_v1_sig = neurons_df_v1[
    (neurons_df_v1["iscell"] == True)
    & neurons_df_v1["is_depth_neuron"]
    & (neurons_df_v1["v1"] == True)
    & (neurons_df_v1["rf_sig"] == True)
]
overview_img = plt.imread(
    flz.get_data_root("processed", flexilims_session=flexilims_session_rf)
    / project_rf
    / "PZAH10.2d/FOVs/PZAH10.2d_overview.tif"
)
overview_pixel_size = 6600 / (2064 / 2)  # um per overview pixel, as in figsupp9

overview_gradients_df = fov_gradients.overview_gradients(
    neurons_df_v1_sig, overview_pixel_size
)

fig = plt.figure(figsize=(15 / 2.54, 6 / 2.54), dpi=300)
fov_gradients.plot_gradients_on_overview(
    fig,
    [rect_cm(fig, 0.3 + i * 5.0, 0.8, 4.5, 4.84) for i in range(3)],
    overview_gradients_df,
    neurons_df_v1_sig,
    xlims=[50, overview_img.shape[1] - 50],
    ylims=[50, overview_img.shape[0] - 50],
    pixel_size=overview_pixel_size,
    fontsize_dict=FONTSIZE_DICT,
)
save_path = SAVE_ROOT / "fig_supp_multidepth_rf" / f"figsupp_fov_gradients_overview{OUT_SFX}"
for ext in ["pdf", "svg", "png"]:
    style.savefig(f"{save_path}.{ext}", fig=fig, dpi=300, bbox_inches="tight")
plt.show()